In [16]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
    TokenValues,
)


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    generic_merge_tables_as_df,
    get_full_table_as_df,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import ALL_CHAINS, ROOT_PRICE_ORACLE, ChainData, STATS_CALCULATOR_REGISTRY, WETH


chain = ALL_CHAINS[0]
possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    DestinationTokenValues,
    DestinationTokenValues.block,
    possible_blocks,
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)

full_destination_df = natural_left_right_using_where(
    DestinationTokens,
    Destinations,
    using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    where_clause=None,
)


# def

full_destination_df
#     DestinationTokens, where_clause=DestinationTokens.chain_id == chain.chain_id
# )
# all_destinations: list[Destinations] = get_full_table_as_orm(
#     Destinations, where_clause=Destinations.chain_id == chain.chain_id
# )

2025-04-25 14:30:08,701 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-25 14:30:08,703 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-25 14:30:08,703 INFO sqlalchemy.engine.Engine [cached since 309.6s ago] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'date_trunc_2': 'day'}
2025-04-25 14:30:08,897 INFO sqlalchemy.engine.Engine COMMIT
2025-04-25 14:30:08,989 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-25 14:30:08,990 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destination_token_values
            WHERE destination_token_values.chain_id = 1
        
2025-04-25 14:30:08,990 INFO sqlalchemy.engine.Engine [cached since 309.6s ago] {}
2025-04-25 14:30:09,179 INFO 

,destination_vault_address,chain_id,token_address,index,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
2,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
3,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38,1,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
4,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,1,0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0,balancer,20756407,Tokemak-Wrapped Ether-Balancer wstETH-WETH Sta...,toke-WETH-wstETH-WETH-BPT,balCompStable,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,wstETH-WETH-BPT,wstETH-WETH-BPT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,0xA94031Ed4b316B043464FDd5482877F42A39845a,8453,0xEDfa23602D0EC14714057867A78d01e94176BEA0,1,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - WETH/wrsETH,toke-WETH-vAMM-WETH/wrsETH,vAMM,0xA24382874A6FD59de45BbccFa160488647514c28,0xA24382874A6FD59de45BbccFa160488647514c28,vAMM-WETH/wrsETH,vAMM-WETH/wrsETH
170,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
171,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x4200000000000000000000000000000000000006,1,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
172,0xBd137c56f3116E5c36753037a784FF844F84F59c,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,balancer,22661893,Tokemak-Wrapped Ether-Gyroscope ECLP cbETH/wstETH,toke-WETH-ECLP-cbETH-wstETH,balGyro,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,ECLP-cbETH-wstETH,ECLP-cbETH-wstETH


In [23]:
def build_pool_token_spot_price_calls(
    chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
) -> list[Call]:

    return [
        Call(
            ROOT_PRICE_ORACLE(chain),
            ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
            [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
        )
        for (pool_address, token_address) in zip(pool_addresses, token_addresses)
    ]


spot_price_calls = build_pool_token_spot_price_calls(
    chain, full_destination_df["pool"], full_destination_df["token_address"]
)
destination_token_spot_price_df = get_raw_state_by_blocks(
    spot_price_calls, possible_blocks, chain, include_block_number=True
)
destination_token_spot_price_df


def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
    return [
        Call(
            dest,
            "underlyingReserves()(address[],uint256[])",
            [
                (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
                (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
            ],
        )
        for dest in destinations
    ]


underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
underlying_reserves_df = get_raw_state_by_blocks(
    underlying_reserves_calls, possible_blocks, chain, include_block_number=True
)
underlying_reserves_df

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]

,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0_underlyingReserves_tokens,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0_underlyingReserves_amounts,0x865e59D439BF7310c9BC6117E6020B8C87De4065_underlyingReserves_tokens,0x865e59D439BF7310c9BC6117E6020B8C87De4065_underlyingReserves_amounts,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1_underlyingReserves_tokens,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1_underlyingReserves_amounts,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2_underlyingReserves_tokens,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2_underlyingReserves_amounts,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1_underlyingReserves_tokens,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1_underlyingReserves_amounts,...,0x945a4f719018edBa445ca67bDa43663C815835Ad_underlyingReserves_amounts,0x58C2233399B85b53C5506f78eAaae9b0DBA1eD3E_underlyingReserves_tokens,0x58C2233399B85b53C5506f78eAaae9b0DBA1eD3E_underlyingReserves_amounts,0xA94031Ed4b316B043464FDd5482877F42A39845a_underlyingReserves_tokens,0xA94031Ed4b316B043464FDd5482877F42A39845a_underlyingReserves_amounts,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD_underlyingReserves_tokens,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD_underlyingReserves_amounts,0xBd137c56f3116E5c36753037a784FF844F84F59c_underlyingReserves_tokens,0xBd137c56f3116E5c36753037a784FF844F84F59c_underlyingReserves_amounts,block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5870693219026985994815, 7077280862498276144414)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(11972633170485222551271, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3559954742803015066641, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(285757389953020082140, 23349768136731744519)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3479229125022580875, 391176699568247377165)",...,None,None,None,None,None,None,None,None,None,20759464
2024-09-16 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5857599679750111099968, 7091767922470441472101)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12059316064033550880768, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3705579922420886348363, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(299080648599456663386, 7653699743623594267)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(6875608924680780947, 387474336757153124511)",...,None,None,None,None,None,None,None,None,None,20766617
2024-09-17 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5902084056519150408600, 7042071304223351542629)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12059930742591700725907, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3932718810709200895813, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(417230185523035338573, 4242219635600785217)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(18655371353584391635, 374617990268654765055)",...,None,None,None,None,None,None,None,None,None,20773761
2024-09-18 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5912023157919524769344, 7145366573925009276361)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12085549571876884432461, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3730067734325149020629, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(414339950273813182559, 7699491265790593224)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(11032004839351892181, 382960258205385395611)",...,None,None,None,None,None,None,None,None,None,20780916
2024-09-19 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5980207145714418730597, 7069241069998441237411)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12023368869347157411194, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b

,destination_vault_address,chain_id,token_address,index,destination_vault_address,chain_id,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,B-rETH-STABLE
2,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,0,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
3,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38,1,0x865e59D439BF7310c9BC6117E6020B8C87De4065,1,balancer,20756406,Tokemak-Wrapped Ether-Balancer osETH/wETH Stab...,toke-WETH-osETH/wETH-BPT,balCompStable,0xDACf5Fa19b1f720111609043ac67A9818262850c,0xDACf5Fa19b1f720111609043ac67A9818262850c,osETH/wETH-BPT,osETH/wETH-BPT
4,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,1,0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0,0,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,1,balancer,20756407,Tokemak-Wrapped Ether-Balancer wstETH-WETH Sta...,toke-WETH-wstETH-WETH-BPT,balCompStable,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,0x93d199263632a4EF4Bb438F1feB99e57b4b5f0BD,wstETH-WETH-BPT,wstETH-WETH-BPT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,0xA94031Ed4b316B043464FDd5482877F42A39845a,8453,0xEDfa23602D0EC14714057867A78d01e94176BEA0,1,0xA94031Ed4b316B043464FDd5482877F42A39845a,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - WETH/wrsETH,toke-WETH-vAMM-WETH/wrsETH,vAMM,0xA24382874A6FD59de45BbccFa160488647514c28,0xA24382874A6FD59de45BbccFa160488647514c28,vAMM-WETH/wrsETH,vAMM-WETH/wrsETH
166,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
167,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,0x4200000000000000000000000000000000000006,1,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD,8453,aerodrome,22326389,Tokemak-Wrapped Ether-Volatile AMM - cbETH/WETH,toke-WETH-vAMM-cbETH/WETH,vAMM,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,0x44Ecc644449fC3a9858d2007CaA8CFAa4C561f91,vAMM-cbETH/WETH,vAMM-cbETH/WETH
168,0xBd137c56f3116E5c36753037a784FF844F84F59c,8453,0x2Ae3F1Ec7F1F5012CFEab0185bfc7aa3cf0DEc22,0,0xBd137c56f3116E5c36753037a784FF844F84F59c,8453,balancer,22661893,Tokemak-Wrapped Ether-Gyroscope ECLP cbETH/wstETH,toke-WETH-ECLP-cbETH-wstETH,balGyro,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,0x54D86E177cdC664B5F9B17eB5fd6A76Fa529e466,ECLP-cbETH-wstETH,ECLP-cbETH-wstETH


[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0_underlyingReserves_tokens,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0_underlyingReserves_amounts,0x865e59D439BF7310c9BC6117E6020B8C87De4065_underlyingReserves_tokens,0x865e59D439BF7310c9BC6117E6020B8C87De4065_underlyingReserves_amounts,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1_underlyingReserves_tokens,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1_underlyingReserves_amounts,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2_underlyingReserves_tokens,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2_underlyingReserves_amounts,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1_underlyingReserves_tokens,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1_underlyingReserves_amounts,...,0x945a4f719018edBa445ca67bDa43663C815835Ad_underlyingReserves_amounts,0x58C2233399B85b53C5506f78eAaae9b0DBA1eD3E_underlyingReserves_tokens,0x58C2233399B85b53C5506f78eAaae9b0DBA1eD3E_underlyingReserves_amounts,0xA94031Ed4b316B043464FDd5482877F42A39845a_underlyingReserves_tokens,0xA94031Ed4b316B043464FDd5482877F42A39845a_underlyingReserves_amounts,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD_underlyingReserves_tokens,0xd18db4dD6aF6A7536aD7F863C136463681e0CdAD_underlyingReserves_amounts,0xBd137c56f3116E5c36753037a784FF844F84F59c_underlyingReserves_tokens,0xBd137c56f3116E5c36753037a784FF844F84F59c_underlyingReserves_amounts,block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5870693219026985994815, 7077280862498276144414)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(11972633170485222551271, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3559954742803015066641, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(285757389953020082140, 23349768136731744519)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3479229125022580875, 391176699568247377165)",...,None,None,None,None,None,None,None,None,None,20759464
2024-09-16 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5857599679750111099968, 7091767922470441472101)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12059316064033550880768, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3705579922420886348363, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(299080648599456663386, 7653699743623594267)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(6875608924680780947, 387474336757153124511)",...,None,None,None,None,None,None,None,None,None,20766617
2024-09-17 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5902084056519150408600, 7042071304223351542629)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12059930742591700725907, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3932718810709200895813, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(417230185523035338573, 4242219635600785217)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(18655371353584391635, 374617990268654765055)",...,None,None,None,None,None,None,None,None,None,20773761
2024-09-18 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5912023157919524769344, 7145366573925009276361)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12085549571876884432461, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(3730067734325149020629, 259614842926764730115...","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(414339950273813182559, 7699491265790593224)","(0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0, 0...","(11032004839351892181, 382960258205385395611)",...,None,None,None,None,None,None,None,None,None,20780916
2024-09-19 23:59:59+00:00,"(0xae78736cd615f374d3085123a210448e74fc6393, 0...","(5980207145714418730597, 7069241069998441237411)","(0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2, 0...","(12023368869347157411194, 25961484292674185776...","(0x7f39c581f595b53c5cb19bd0b

In [22]:
destination_token_spot_price_df

NameError: name 'destination_token_spot_price_df' is not defined

In [ ]:
token_value_df = get_full_table_as_df(TokenValues, where_clause=TokenValues.chain_id == chain.chain_id)
# (token_address, chain_id, block) -> [safe_price, backing]
token_value_df

2025-04-25 11:44:50,683 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-25 11:44:50,684 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-25 11:44:50,685 INFO sqlalchemy.engine.Engine [cached since 844.6s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x13acf70a0>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-25 11:44:50,686 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM token_values
            WHERE token_values.chain_id = 1
        
2025-04-25 11:44:50,68

,block,chain_id,token_address,denomiated_in,backing,safe_price
0,20759464,1,0xCAcd6fd266aF91b8AeD52aCCc382b4e165586E29,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
1,20759464,1,0xdAC17F958D2ee523a2206206994597C13D831ec7,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
2,20759464,1,0xC71Ea051a5F82c67ADcF634c36FFE6334793D24C,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN
3,20759464,1,0x5E8422345238F34275888049021821E8E08CAa1f,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.999145
4,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.047726
...,...,...,...,...,...,...
6433,22342307,1,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000566
6434,22342307,1,0x83F20F44975D03b1b09e64809B757c47f942BEeA,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000654
6435,22342307,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000566
6436,22342307,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.041302


In [ ]:
a = DestinationTokenValues()

TypeError: __init__() missing 8 required positional arguments: 'block', 'chain_id', 'token_address', 'destination_address', 'spot_price', 'quantity', 'safe_spot_spread', and 'spot_backing_discount'